# Configuration

In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
from tqdm import tqdm
from os import urandom
from math import log2

WORD_SIZE = 16
MASK_VAL = 2 ** WORD_SIZE - 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Model

In [2]:
class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv1d(channels, 32, kernel_size, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, channels, kernel_size, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        res = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = x + res
        return x


class ImprovedCNN(nn.Module):
    def __init__(self, depth=10):
        super(ImprovedCNN, self).__init__()
        self.in_channels = 3
        self.res_tower = nn.Sequential(*[ResBlock(self.in_channels) for _ in range(depth)])
        self.fc1 = nn.Linear(3 * 16, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.out = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.res_tower(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.sigmoid(self.out(x))
        return x


# Crypto Helpers

In [3]:
def expand_key(key, rounds):
    """Speck32/64 Key Schedule"""
    k = [key[i] for i in range(4)]
    ks = [0] * rounds
    ks[0] = k[0]
    l = k[1:]
    for i in range(rounds - 1):
        l_curr = l[0]
        l_curr = ((l_curr >> 7) | (l_curr << 9)) & MASK_VAL
        l_curr = (int(l_curr) + int(ks[i])) & MASK_VAL
        l_curr = l_curr ^ i
        l = l[1:] + [l_curr]
        k_curr = ks[i]
        k_curr = ((k_curr << 2) | (k_curr >> 14)) & MASK_VAL
        k_curr = k_curr ^ l_curr
        ks[i + 1] = k_curr
    return ks


def encrypt_vectorized(p_left, p_right, ks):
    """Numpy Vectorized Encryption"""
    x = p_left.astype(np.int32)
    y = p_right.astype(np.int32)
    for k in ks:
        x = ((x >> 7) | (x << 9)) & MASK_VAL
        x = (x + y) & MASK_VAL
        x = x ^ k
        y = ((y << 2) | (y >> 14)) & MASK_VAL
        y = y ^ x
    return x.astype(np.uint16), y.astype(np.uint16)


def dec_one_round_torch(c0, c1, k):
    """PyTorch Decryption (One Round)"""
    if c0.dim() == 1: c0 = c0.unsqueeze(1)
    if c1.dim() == 1: c1 = c1.unsqueeze(1)
    if k.dim() == 1: k = k.unsqueeze(0)

    # Invert Round: y = ROR(y^x, 2); x = ROL((x^k)-y, 7)
    c1 = c1 ^ c0
    c1 = ((c1 >> 2) | (c1 << 14)) & MASK_VAL

    c0 = c0 ^ k
    if c1.shape != c0.shape:
        c1 = c1.expand_as(c0)

    c0 = (c0 - c1) & MASK_VAL
    c0 = ((c0 << 7) | (c0 >> 9)) & MASK_VAL
    return c0, c1


def extract_features_torch(c0l, c0r, c1l, c1r):
    """Features: (c0l^c1l, c0l^c0r, c0l^c1r)"""
    f1 = c0l ^ c1l
    f2 = c0l ^ c0r
    f3 = c0l ^ c1r
    bits = torch.arange(16, device=c0l.device).reshape(1, 1, 16)
    f_stack = torch.stack([f1, f2, f3], dim=1).unsqueeze(2)
    X = (f_stack >> bits) & 1
    return X.float()


def make_structure(pt0, pt1, diff=(0x0040, 0x0000), neutral_bits=[20, 21, 22, 14, 15]):
    """Gohr's Neutral Bit Structure Generation"""
    p0l = np.array([pt0], dtype=np.uint16)
    p0r = np.array([pt1], dtype=np.uint16)
    for i in neutral_bits:
        d = 1 << i
        d_high = (d >> 16) & 0xFFFF
        d_low = d & 0xFFFF
        p0l = np.concatenate([p0l, p0l ^ d_high])
        p0r = np.concatenate([p0r, p0r ^ d_low])
    p1l = p0l ^ diff[0]
    p1r = p0r ^ diff[1]
    return [(p0l, p0r, p1l, p1r)]



# WKRP

In [4]:
def generate_full_wkrp(model, dist_rounds, n_samples=2000):
    """
    Generates the Full WKRP Table (65536 entries).
    Simulates: Encrypt 1 round (k=0) -> Decrypt 1 round (k=diff)
    """
    print(f"Generating WKRP for {dist_rounds}r DND (using {n_samples} samples/diff)...")
    model.eval()

    # 1. Generate base data (Target State)
    dummy_key = np.zeros(8, dtype=np.uint16)
    ks_dist = expand_key(dummy_key, dist_rounds)

    c0l_list, c0r_list, c1l_list, c1r_list = [], [], [], []
    batches = (n_samples // 32) + 1

    for _ in range(batches):
        pt0 = np.random.randint(0, 65536, dtype=np.uint16)
        pt1 = np.random.randint(0, 65536, dtype=np.uint16)
        struct = make_structure(pt0, pt1)[0]

        # Encrypt to Round R (distinguisher input)
        c0l, c0r = encrypt_vectorized(struct[0], struct[1], ks_dist)
        c1l, c1r = encrypt_vectorized(struct[2], struct[3], ks_dist)

        # Encrypt 1 EXTRA Round with Key=0 (Attack input)
        # x = (ROR(x,7) + y)^k; y = ROL(y,2)^x
        x, y = c0l.astype(np.int32), c0r.astype(np.int32)
        x = ((x >> 7) | (x << 9)) & MASK_VAL
        x = (x + y) & MASK_VAL  # k=0
        y = ((y << 2) | (y >> 14)) & MASK_VAL
        y = y ^ x
        c0l, c0r = x.astype(np.uint16), y.astype(np.uint16)

        x, y = c1l.astype(np.int32), c1r.astype(np.int32)
        x = ((x >> 7) | (x << 9)) & MASK_VAL
        x = (x + y) & MASK_VAL  # k=0
        y = ((y << 2) | (y >> 14)) & MASK_VAL
        y = y ^ x
        c1l, c1r = x.astype(np.uint16), y.astype(np.uint16)

        c0l_list.append(c0l)
        c0r_list.append(c0r)
        c1l_list.append(c1l)
        c1r_list.append(c1r)

    # Flatten
    c0l = np.concatenate(c0l_list)[:n_samples]
    c0r = np.concatenate(c0r_list)[:n_samples]
    c1l = np.concatenate(c1l_list)[:n_samples]
    c1r = np.concatenate(c1r_list)[:n_samples]

    # To GPU
    c0l = torch.tensor(c0l, dtype=torch.int32, device=DEVICE).unsqueeze(1)
    c0r = torch.tensor(c0r, dtype=torch.int32, device=DEVICE).unsqueeze(1)
    c1l = torch.tensor(c1l, dtype=torch.int32, device=DEVICE).unsqueeze(1)
    c1r = torch.tensor(c1r, dtype=torch.int32, device=DEVICE).unsqueeze(1)

    # 2. Profile Diffs
    all_diffs = torch.arange(0, 65536, dtype=torch.int32, device=DEVICE)
    mu_table = torch.zeros(65536, device=DEVICE)
    std_table = torch.zeros(65536, device=DEVICE)

    diff_batch = 512
    with torch.no_grad():
        for i in tqdm(range(0, 65536, diff_batch), desc="WKRP Profiling"):
            batch = all_diffs[i: i + diff_batch]
            d0l, d0r = dec_one_round_torch(c0l, c0r, batch)
            d1l, d1r = dec_one_round_torch(c1l, c1r, batch)

            X = extract_features_torch(d0l.flatten(), d0r.flatten(), d1l.flatten(), d1r.flatten())
            v = model(X).squeeze()

            eps = 1e-7
            v = torch.clamp(v, eps, 1 - eps)
            z = torch.log2(v / (1 - v))

            z_mat = z.view(c0l.size(0), -1)
            mu_table[i:i + diff_batch] = z_mat.mean(dim=0)
            std_table[i:i + diff_batch] = z_mat.std(dim=0)

    return mu_table, std_table


# Bayesian Key Search

In [5]:
def improved_bayesian_key_search(ciphertext_structures, model, wkrp_mean, wkrp_std, n_candidates=32, n_iterations=5,
                                 best_key_hint=None):
    """
    Algorithm 1: Improved Bayesian Key Search
    Includes Dynamic Key Retention (Global Best Key)
    """
    model.eval()

    # 1. Preprocess Structures
    c0l_list = [s[0] for s in ciphertext_structures]
    c0r_list = [s[1] for s in ciphertext_structures]
    c1l_list = [s[2] for s in ciphertext_structures]
    c1r_list = [s[3] for s in ciphertext_structures]

    # Flatten all pairs: (N_total_pairs, 1)
    c0l = torch.tensor(np.concatenate(c0l_list), dtype=torch.int32, device=DEVICE).unsqueeze(1)
    c0r = torch.tensor(np.concatenate(c0r_list), dtype=torch.int32, device=DEVICE).unsqueeze(1)
    c1l = torch.tensor(np.concatenate(c1l_list), dtype=torch.int32, device=DEVICE).unsqueeze(1)
    c1r = torch.tensor(np.concatenate(c1r_list), dtype=torch.int32, device=DEVICE).unsqueeze(1)

    n_pairs = c0l.size(0)
    KEY_SPACE = 65536
    all_keys_indices = torch.arange(KEY_SPACE, device=DEVICE, dtype=torch.long)

    if not torch.is_tensor(wkrp_mean): wkrp_mean = torch.tensor(wkrp_mean, device=DEVICE)
    if not torch.is_tensor(wkrp_std): wkrp_std = torch.tensor(wkrp_std, device=DEVICE)

    # [cite_start]2. Initialization (Step 1-2) [cite: 311-312]
    S_rand = torch.randperm(KEY_SPACE, device=DEVICE)
    if best_key_hint is not None:
        S = torch.cat([torch.tensor([best_key_hint], device=DEVICE), S_rand])[:n_candidates]
    else:
        S = S_rand[:n_candidates]

    L = []

    # [cite_start]TRACKING VARIABLES FOR RETENTION [cite: 175]
    global_best_key = best_key_hint
    global_best_score = -float('inf')

    # [cite_start]3. Main Loop [cite: 314]
    for t in range(n_iterations):
        # Decrypt candidates (Step 7)
        d0l, d0r = dec_one_round_torch(c0l, c0r, S)
        d1l, d1r = dec_one_round_torch(c1l, c1r, S)

        # Expand d0r and d1r to match the batch dimension
        # d0r is (Pairs, 1), we need (Pairs, n_cand) to match d0l which is (Pairs, n_cand)
        n_curr_cand = S.size(0)
        if d0r.shape[1] != n_curr_cand: d0r = d0r.expand(-1, n_curr_cand)
        if d1r.shape[1] != n_curr_cand: d1r = d1r.expand(-1, n_curr_cand)

        # Predict (Step 8)
        X = extract_features_torch(d0l.flatten(), d0r.flatten(), d1l.flatten(), d1r.flatten())
        with torch.no_grad():
            v = model(X).squeeze()

        # Log-Odds (Step 9)
        epsilon = 1e-7
        v = torch.clamp(v, epsilon, 1 - epsilon)
        z = torch.log2(v / (1 - v))

        # Stats
        z_matrix = z.view(n_pairs, n_curr_cand)
        s_ki = torch.sum(z_matrix, dim=0)  # Total Score [cite: 338]
        m_ki = torch.mean(z_matrix, dim=0)  # Mean Log-Odds [cite: 340]

        # Update History L
        curr_keys_np = S.cpu().numpy()
        curr_scores_np = s_ki.cpu().numpy()
        for k, sc in zip(curr_keys_np, curr_scores_np):
            L.append((int(k), float(sc)))

        # [cite_start]--- GLOBAL BEST KEY UPDATE [cite: 175] ---
        batch_max_idx = torch.argmax(s_ki)
        batch_max_score = s_ki[batch_max_idx]
        batch_best_key = S[batch_max_idx].item()

        if batch_max_score > global_best_score:
            global_best_score = batch_max_score
            global_best_key = batch_best_key

        # [cite_start]Bayesian Update (Step 16-18) [cite: 343]
        # lambda(k) = sum( (m - mu)^2 / sigma^2 )
        S_exp = S.unsqueeze(1).long()
        K_exp = all_keys_indices.unsqueeze(0)
        diffs = S_exp ^ K_exp

        mus = wkrp_mean[diffs]
        sigs = wkrp_std[diffs]

        m_exp = m_ki.unsqueeze(1)
        numer = (m_exp - mus) ** 2
        denom = sigs ** 2 + 1e-9
        lambdas = torch.sum(numer / denom, dim=0)

        # Select new S
        _, top_k_indices = torch.topk(lambdas, n_candidates, largest=False)
        S = top_k_indices.int()

        # [cite_start]--- FORCE RETENTION [cite: 175] ---
        # "Retains the best key and adds it to the new candidate key."
        if global_best_key is not None:
            if not (S == global_best_key).any():
                S[-1] = int(global_best_key)

    # Return results
    results = {}
    for k, s in L:
        if k not in results or s > results[k]:
            results[k] = s

    final_L = sorted(results.items(), key=lambda x: x[1], reverse=True)
    return final_L


# -----------------------------------------------------------
# 5. Test Harness (Similar to test_key_recovery.py)
# -----------------------------------------------------------
def gen_challenge(n_structures, rounds, neutral_bits=[20, 21, 22, 14, 15]):
    """Generates the challenge data (Structures)"""
    secret_key = np.frombuffer(urandom(8), dtype=np.uint16)
    ks = expand_key(secret_key, rounds)
    target_subkey = ks[-1]

    structures = []
    for _ in range(n_structures):
        pt0 = np.random.randint(0, 65536, dtype=np.uint16)
        pt1 = np.random.randint(0, 65536, dtype=np.uint16)
        struct = make_structure(pt0, pt1, neutral_bits=neutral_bits)[0]
        c0l, c0r = encrypt_vectorized(struct[0], struct[1], ks)
        c1l, c1r = encrypt_vectorized(struct[2], struct[3], ks)
        structures.append((c0l, c0r, c1l, c1r))

    return structures, target_subkey


def test(model, dist_rounds, attack_rounds, n_trials=20, n_structures=10, top_k=32):
    print(f"--- Improved Bayesian Key Search Attack ---")
    print(f"Distinguisher: {dist_rounds}r | Attack: {attack_rounds}r | Structures: {n_structures}")

    # Generate WKRP
    wkrp_mu, wkrp_std = generate_full_wkrp(model, dist_rounds, n_samples=2000)

    hits = 0
    ranks = []

    for i in tqdm(range(n_trials), desc="Trials"):
        # 1. Generate Challenge
        structures, real_subkey = gen_challenge(n_structures, attack_rounds)

        # 2. Run Improved BKS
        candidates = improved_bayesian_key_search(
            structures, model, wkrp_mu, wkrp_std,
            n_candidates=top_k, n_iterations=5
        )

        # 3. Check Result
        predicted_keys = [k for k, s in candidates]

        # Success if real key is in top_k unique guesses
        # (Note: candidates list from BKS is already unique-filtered)
        if real_subkey in predicted_keys[:top_k]:
            hits += 1
            rank = predicted_keys.index(real_subkey) + 1
            ranks.append(rank)

    print(f"\nFinal Results:")
    print(f"Success Rate: {(hits / n_trials) * 100:.2f}%")
    print(f"Avg Rank: {np.mean(ranks) if ranks else 0:.2f}")


# Main

In [6]:
SCENARIOS = {
        5: {"dist_rounds": 5, "attack_rounds": 6, "depth": 10, "n_structures": 32},
        6: {"dist_rounds": 6, "attack_rounds": 7, "depth": 10, "n_structures": 64},
        7: {"dist_rounds": 7, "attack_rounds": 8, "depth": 1, "n_structures": 128},
        8: {"dist_rounds": 8, "attack_rounds": 9, "depth": 1, "n_structures": 256},
    }

for r in SCENARIOS:
    cfg = SCENARIOS[r]
    model_path = f"light_models/dnd_speck32_r{r}_d{cfg['depth']}.pt"

    if not os.path.exists(model_path):
        print(f"Skipping {model_path} (File not found)")
        continue

    print(f"\nLoading Model: {model_path}")
    try:
        model = ImprovedCNN(depth=cfg['depth']).to(DEVICE)
        checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
    except Exception as e:
        print(f"Failed to load model: {e}")
        continue

    test(
        model=model,
        dist_rounds=cfg['dist_rounds'],
        attack_rounds=cfg['attack_rounds'],
        n_trials=10,
        n_structures=cfg['n_structures'],
        top_k=32
    )


Loading Model: light_models/dnd_speck32_r5_d10.pt
--- Improved Bayesian Key Search Attack ---
Distinguisher: 5r | Attack: 6r | Structures: 32
Generating WKRP for 5r DND (using 2000 samples/diff)...


Trials: 100%|██████████| 10/10 [00:28<00:00,  2.87s/it]



Final Results:
Success Rate: 100.00%
Avg Rank: 1.00

Loading Model: light_models/dnd_speck32_r6_d10.pt
--- Improved Bayesian Key Search Attack ---
Distinguisher: 6r | Attack: 7r | Structures: 64
Generating WKRP for 6r DND (using 2000 samples/diff)...


Trials: 100%|██████████| 10/10 [00:36<00:00,  3.65s/it]



Final Results:
Success Rate: 100.00%
Avg Rank: 1.00

Loading Model: light_models/dnd_speck32_r7_d1.pt
--- Improved Bayesian Key Search Attack ---
Distinguisher: 7r | Attack: 8r | Structures: 128
Generating WKRP for 7r DND (using 2000 samples/diff)...


Trials: 100%|██████████| 10/10 [00:10<00:00,  1.07s/it]



Final Results:
Success Rate: 20.00%
Avg Rank: 2.50

Loading Model: light_models/dnd_speck32_r8_d1.pt
--- Improved Bayesian Key Search Attack ---
Distinguisher: 8r | Attack: 9r | Structures: 256
Generating WKRP for 8r DND (using 2000 samples/diff)...


Trials: 100%|██████████| 10/10 [00:12<00:00,  1.28s/it]


Final Results:
Success Rate: 0.00%
Avg Rank: 0.00
